# Hidden Rhythms: What the FFT Actually Tells an Analytical Chemist
### 3.7 — Frequency Domain: A Practical Look at the FFT

A pump pulses. Mains electricity hums at 50 or 60 Hz. A lab's air handling cycles
through the day. None of these are the chemistry you're measuring, yet every one of
them can ride along on your signal as a faint, repeating ripple — and in the raw
trace they look like vague wobble you can't quite pin down.

The Fourier transform turns that vague wobble into something you can *point at*. It
re-plots your signal with a different x-axis: instead of "value versus time," it shows
"how much of each repeating rhythm is present versus frequency." A periodic
interference that's invisible in the time trace becomes a single sharp peak whose
position fingerprints its source.

This is a practical, measurement-first look at the FFT — what it tells you, how to
read it, and the two ways it can quietly lie. We skip the derivations; the goal is
judgement, not maths.

> **One idea to hold onto:** a signal in time and the same signal in frequency are two
> views of one thing. The FFT swaps the x-axis from *when* to *how often* — turning an
> invisible periodic interference into a sharp, locatable peak.

**By the end of this notebook you will be able to:**

1. Read a signal in both the time and frequency domains, and move between them.
2. Use the FFT to find a periodic interference (a pump, mains hum, a daily cycle) hiding under noise.
3. Recognize and avoid the two traps that make an FFT mislead you: aliasing and spectral leakage.

## 1. A signal with rhythms hidden inside it

Imagine a detector reading logged over two seconds. On top of the (here, flat)
chemistry sit two periodic interferences: a slow **5 Hz pump pulsation** and a
**50 Hz mains pickup**, plus random noise. We build it from known parts so we can
check whether the FFT recovers them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (6.8, 3.4), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10})

# Acquisition settings -- these matter as much as the chemistry.
fs = 500.0                 # sampling rate: 500 readings per second (Hz)
duration = 2.0             # seconds of data
t = np.arange(0, duration, 1 / fs)     # the time axis (s)

# Two periodic interferences, each (frequency_Hz, amplitude).
PUMP = (5.0, 0.6)          # a 5 Hz pump pulsation
MAINS = (50.0, 0.3)        # 50 Hz electrical pickup

rng = np.random.default_rng(3)
signal = (PUMP[1] * np.sin(2 * np.pi * PUMP[0] * t)
          + MAINS[1] * np.sin(2 * np.pi * MAINS[0] * t)
          + rng.normal(0, 0.25, size=t.size))    # detector noise

print(f"sampled {t.size} points at {fs:.0f} Hz over {duration:.0f} s")
print(f"hidden inside: a {PUMP[0]:.0f} Hz pump and {MAINS[0]:.0f} Hz mains pickup")

## 2. The time-domain view: you can tell *something* repeats

First, plot the signal the way the instrument gives it to you — value versus time.
You can see it's busy and periodic, but try to read off *which* frequencies are
present, or separate the pump from the mains. It's nearly impossible by eye.

In [ ]:
fig, ax = plt.subplots()
ax.plot(t, signal, lw=0.8, color="#1a73e8")
ax.set_xlim(0, 0.5)                  # zoom to half a second so the wiggles are visible
ax.set_xlabel("time (s)")
ax.set_ylabel("detector signal (a.u.)")
ax.set_title("Time domain: clearly periodic, but which frequencies?")
fig.tight_layout()
plt.show()

There's structure in there — a fast ripple riding on a slower swell — but the time
trace won't tell you "5 Hz and 50 Hz." That is exactly the question the frequency
domain answers directly.

## 3. The FFT: reveal the hidden frequencies

The FFT asks, for every frequency, "how much of *this* rhythm is in the signal?" We
use `np.fft.rfft` (the real-input FFT) and `np.fft.rfftfreq` to get the matching
frequency axis. Scaling the magnitudes by `2/N` makes each peak's height read in the
same units as the sinusoid's amplitude, so the plot is directly interpretable.

In [ ]:
N = t.size
spectrum = np.fft.rfft(signal)                 # complex Fourier coefficients
freqs = np.fft.rfftfreq(N, d=1 / fs)           # the frequency axis (Hz)
amplitude = (2 / N) * np.abs(spectrum)         # scaled so peak height ~ sinusoid amplitude

fig, ax = plt.subplots()
ax.plot(freqs, amplitude, lw=1.2, color="#188038")
ax.set_xlim(0, 80)
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("amplitude (a.u.)")
ax.set_title("Frequency domain: two sharp peaks where the time trace had none")
for f0, amp in (PUMP, MAINS):
    ax.annotate(f"{f0:.0f} Hz", xy=(f0, amp), xytext=(f0 + 3, amp + 0.02),
                fontsize=9, color="#5f6368")
fig.tight_layout()
plt.show()

# Read the peaks back out: the two tallest points of the spectrum.
peak_order = np.argsort(amplitude)[::-1]
top2 = sorted(freqs[peak_order[:2]])
print("two strongest frequencies found:", [round(f, 1) for f in top2], "Hz")
print("(we built in 5 Hz and 50 Hz -- recovered directly from the noisy trace)")

Two clean peaks, sitting on the noise floor, exactly at 5 and 50 Hz. The noise that
dominated the time trace spreads thinly across all frequencies here, so the coherent
periodic components stand up sharply above it. **This is the FFT's core value: it
concentrates a repeating signal into a peak you can locate and measure.**

## 4. What "frequency" and "period" mean physically

A peak's position is a **frequency** `f`, in cycles per second (Hz). Its reciprocal is
the **period** `T = 1/f`, the time for one cycle. Reading the peaks as physics is how
you identify the culprit:

In [ ]:
for label, (f0, _) in [("pump pulsation", PUMP), ("mains pickup", MAINS)]:
    print(f"{label:15s}: {f0:5.1f} Hz  ->  period {1/f0*1000:6.1f} ms per cycle")

The 50 Hz peak with its 20 ms period is the unmistakable signature of mains
electricity (it would be 60 Hz in North America). The 5 Hz / 200 ms peak matches a
pump's stroke rate. The FFT didn't tell us *what* caused each peak — **we** did, by
mapping a measured frequency onto knowledge of the instrument. That interpretive step
is the whole point: the transform locates the rhythm; the chemist names it.

## 5. Trap 1 — aliasing: when a peak lands at the wrong frequency

The FFT can only represent frequencies up to **half the sampling rate** (the *Nyquist*
limit, `fs/2`). Sample too slowly and a high-frequency component doesn't vanish — it
**folds back** and shows up as a *lower* frequency that was never really there. This is
aliasing, and it's dangerous precisely because the fake peak looks perfectly real.

Here we sample the same 50 Hz mains signal at only 80 Hz (Nyquist = 40 Hz, below 50).

In [ ]:
fs_slow = 80.0                                  # too slow: Nyquist = 40 Hz < 50 Hz
t_slow = np.arange(0, duration, 1 / fs_slow)
mains_only = np.sin(2 * np.pi * 50.0 * t_slow)  # a pure 50 Hz signal

spec_slow = (2 / t_slow.size) * np.abs(np.fft.rfft(mains_only))
freqs_slow = np.fft.rfftfreq(t_slow.size, d=1 / fs_slow)

apparent = freqs_slow[np.argmax(spec_slow)]
print(f"true frequency        : 50 Hz")
print(f"sampling rate         : {fs_slow:.0f} Hz  (Nyquist = {fs_slow/2:.0f} Hz)")
print(f"frequency the FFT shows: {apparent:.0f} Hz   <- an ALIAS, not real")

fig, ax = plt.subplots()
ax.plot(freqs_slow, spec_slow, lw=1.3, color="#ea4335")
ax.axvline(apparent, ls="--", color="#5f6368", lw=1)
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("amplitude (a.u.)")
ax.set_title(f"Undersampled: 50 Hz masquerades as {apparent:.0f} Hz")
fig.tight_layout()
plt.show()

The 50 Hz mains shows up as a confident peak near 30 Hz (`|50 − 80| = 30`) — a
frequency that does not exist in the sample. If you trusted it, you'd hunt for a
30 Hz source that isn't there. **The defence is in the acquisition, not the analysis:
sample at more than twice the highest frequency you care about** (and filter out
anything faster before it reaches the detector). An FFT can only be as honest as the
sampling that fed it.

## 6. Trap 2 — spectral leakage, and windowing

The FFT quietly assumes your record repeats forever, end joined seamlessly to start.
When a signal contains a whole number of cycles in the window, that's true and the
peak is sharp. When it doesn't, the wrap-around creates a discontinuity, and the
peak's energy **smears** into neighbouring frequencies — *spectral leakage*. A
**window** (here, a Hann window) tapers the record to zero at both ends, removing the
discontinuity and suppressing the smear.

In [ ]:
# A 5.25 Hz tone: it does NOT complete a whole number of cycles in 2 s, so it leaks.
leaky = np.sin(2 * np.pi * 5.25 * t)

def amp_db(y):
    "Magnitude spectrum in dB, normalized to its own peak (so we compare shapes)."
    m = np.abs(np.fft.rfft(y))
    return 20 * np.log10(m / m.max() + 1e-12)

rect_db = amp_db(leaky)                          # rectangular = no window
hann_db = amp_db(leaky * np.hanning(t.size))     # Hann-windowed

fig, ax = plt.subplots()
ax.plot(freqs, rect_db, lw=1.2, color="#ea4335", label="no window (leakage)")
ax.plot(freqs, hann_db, lw=1.2, color="#188038", label="Hann window")
ax.set_xlim(0, 15)
ax.set_ylim(-80, 5)
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("relative magnitude (dB)")
ax.set_title("Spectral leakage: a window tames the smearing skirts")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

Without a window (red) the 5.25 Hz peak spills broad skirts across the whole band — a
weak neighbouring peak could easily hide under them. The Hann window (green) pulls
those skirts down by tens of dB, at the cost of a slightly wider main peak. That trade
— a little resolution for far less leakage — is why windowing is the default for real
spectra. **Rule of thumb: window before you FFT unless you have a reason not to.**

## 7. Where this pays off in the lab

The same tool flags a whole family of analytical problems, because each leaves a
periodic fingerprint at a frequency you can predict and check:

- **Instrument oscillations** — thermal or optical cycling that puts a slow ripple on
  a baseline shows up as a low-frequency peak.
- **Pumps** — HPLC or peristaltic pulsation appears at the stroke frequency (and its
  harmonics); spotting it tells you the baseline noise is mechanical, not chemical.
- **Periodic contamination** — a repeating artifact (a chopper, a scanning mirror, a
  duty-cycled lamp) sits at its own characteristic frequency.
- **Daily rhythms in sensor data** — environmental and process sensors logged over
  days carry a 24-hour cycle from temperature, occupancy, or HVAC.

Here's that last one: a sensor logged every 30 minutes for four days, with a slow
drift, a real 24-hour cycle, and noise. The FFT (in cycles per day) isolates the
daily rhythm as a peak at exactly 1 / day.

In [ ]:
# 4 days sampled every 0.5 h.
dt_h = 0.5
t_h = np.arange(0, 4 * 24, dt_h)                 # time in hours
sensor = (0.004 * t_h                            # slow drift
          + 1.0 * np.sin(2 * np.pi * t_h / 24.0) # a 24-hour cycle
          + rng.normal(0, 0.3, size=t_h.size))   # noise

spec = (2 / t_h.size) * np.abs(np.fft.rfft(sensor - sensor.mean()))  # remove DC offset
cyc_per_day = np.fft.rfftfreq(t_h.size, d=dt_h) * 24.0               # cycles/hour -> /day

fig, (axA, axB) = plt.subplots(1, 2, figsize=(11, 3.4))
axA.plot(t_h / 24, sensor, lw=0.9, color="#1a73e8")
axA.set_xlabel("time (days)"); axA.set_ylabel("sensor reading")
axA.set_title("Sensor log: drift + daily cycle + noise")
axB.plot(cyc_per_day, spec, lw=1.3, color="#188038")
axB.set_xlim(0, 5); axB.axvline(1.0, ls="--", color="#5f6368", lw=1)
axB.set_xlabel("frequency (cycles per day)"); axB.set_ylabel("amplitude")
axB.set_title("FFT: the daily rhythm is a peak at 1 / day")
fig.tight_layout()
plt.show()

print("strongest cycle:", round(cyc_per_day[np.argmax(spec)], 2), "per day (a 24-hour rhythm)")

The drift shows as energy piling up near zero frequency (a very slow "cycle"), and the
24-hour rhythm stands out cleanly at 1 / day — letting you separate a daily
environmental effect from a genuine trend or a chemical change. Same transform, same
reading: *a frequency, mapped to a physical cause.*

## 8. Reading an FFT is interpretation, not arithmetic

The FFT is a lens, not an answer. It reliably tells you **which periodic rhythms are
present and how strong** — but turning that into a conclusion is measurement judgement:

- **You name the source.** The transform gives a frequency; you identify it as mains,
  a pump, or a daily cycle using what you know about the instrument and its environment.
- **A peak is only as trustworthy as the sampling.** If you didn't sample faster than
  twice your highest real frequency, a peak may be an alias sitting at a fake location.
- **Mind leakage before reading peak heights or close-spaced peaks.** Window first.
- **Absence of a peak isn't absence of a cause** — a drifting or one-off disturbance
  isn't periodic and won't appear as a peak; the FFT is the wrong tool for those.

Used this way, the frequency domain answers a question the time trace can't: *is the
junk on my signal periodic, and if so, what in the lab is causing it?*

## Key Takeaways

- The FFT re-plots a signal from *time* to *frequency*, concentrating a repeating
  component into a **locatable peak** whose position is its frequency (`period = 1/f`).
- It excels at finding **periodic interference** — pumps, mains hum, instrument
  oscillations, daily sensor rhythms — that's invisible in the time trace.
- **Aliasing:** undersampling makes a high frequency appear at a false low one. Sample
  faster than twice your highest real frequency (the Nyquist rule).
- **Leakage:** a non-integer number of cycles smears a peak; a **window** (e.g. Hann)
  suppresses the smear for a small resolution cost.
- Reading an FFT is **interpretation** — the transform locates a rhythm; the chemist
  names its cause.

## Practical Checklist

- [ ] Record the **sampling rate**; confirm it exceeds twice your highest frequency of interest.
- [ ] Remove the DC offset (subtract the mean) so a large baseline doesn't dominate.
- [ ] **Window** the record (Hann is a safe default) before transforming.
- [ ] Read peak positions as frequencies, convert to periods, and map them to physical sources.
- [ ] Sanity-check suspicious low-frequency peaks for aliasing before believing them.

## Common Mistakes

- Trusting a peak from **undersampled** data — it may be an alias at the wrong frequency.
- Skipping the **window** and mistaking leakage skirts for real neighbouring peaks.
- Forgetting to remove the **DC/baseline** offset, which buries low-frequency detail.
- Treating the FFT as automatic — it locates rhythms but does **not** identify causes,
  and it says nothing useful about non-periodic drift or one-off events.

## Reporting Guidance

- State the **sampling rate**, record length (which sets the frequency resolution
  `1/T`), and any **window** used — they determine which frequencies an FFT can show
  and how sharply, and are part of how the result was produced.

## Next Lesson

You've reached the end of the core **Track 3 — Signal Processing** arc: noise and why
preprocessing exists, smoothing, baselines, peak detection and integration, signal
averaging, and now the frequency domain. From here the curriculum branches into
technique-specific work — **Track 4 — Spectroscopy**, starting with **4.1 — Beer–Lambert
and the Absorbance Mindset** — where this clean, well-understood signal becomes
quantitative chemistry. (An advanced **Peak Fitting** lesson, 3.8, remains as optional
further Track 3 material.)